# 01_train_yolo

Generated by `notebooks/build_notebooks.py`. Runtime → Change runtime type → GPU, then Run all.

In [ ]:
# @title Setup: GPU check + repo + deps

In [ ]:
!nvidia-smi

In [ ]:
from pathlib import Path
REPO='/content/repo'
!git clone https://github.com/arupa444/trainDronisight.git $REPO 2>/dev/null || (cd $REPO && git pull)
%cd $REPO
!pip -q install uv && uv pip install --system -e .

In [ ]:
import torch; print('CUDA:', torch.cuda.is_available())

In [ ]:
from notebooks.colab_utils import mount_drive, drive_db_zip, ensure_dataset
mount_drive()

In [ ]:
import os; os.environ['DRONISIGHT_DATA'] = '/content/data'  # matches the ensure_dataset() dest below

In [ ]:
ensure_dataset(drive_db_zip('yolo_train_db'), '/content/data', 'yolo_train_db')

In [ ]:
# 1) pole: fills the frame -> imgsz 640 (matches train_pole.py default)
!python -m train_yolo.train_pole --version clahe --epochs 100 --imgsz 640 --batch 16

In [ ]:
# 2) component_above_1000 (wire/h_insulator/v_insulator/crossarm_stright): 1280 for thin wires
!python -m train_yolo.train_components --subset component_above_1000 --version clahe --epochs 150 --imgsz 1280 --batch 16 --model yolo26m.pt

In [ ]:
# 3) component_below_1000 (vegetation/top_crossarm/om_crossarm/rust): oversampled train, more epochs
!python -m train_yolo.train_components --subset component_below_1000 --version clahe --epochs 200 --imgsz 1280 --batch 16 --model yolo26m.pt

In [ ]:
# 4) component_classification (14 condition classes; train balanced to ~400/class)
!python -m train_yolo.train_components --subset component_classification --version clahe --epochs 150 --imgsz 1280 --batch 16 --model yolo26m.pt

In [ ]:
# Colab runtimes are ephemeral -> copy weights + plots to Drive before the session ends
from notebooks.colab_utils import save_runs_to_drive
print('saved to', save_runs_to_drive())